# Hand-Pose Confidence Estimation under Occlusion — GPU run

Runs the real experiment: **HaMeR** on **HO-3D v3**, four uncertainty estimators,
occlusion-stratified analysis. The analysis pipeline is already validated end-to-end
on synthetic data (`scripts/synthetic_validation.py`).

**Before running:** Runtime → Change runtime type → **GPU** (T4 is fine, A100 faster).



In [ ]:
# 0. GPU + Drive
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 1. Get the project code (pick ONE)
# Option A: from Drive
!cp -r /content/drive/MyDrive/cs231n-project /content/proj
# Option B: from GitHub (if you pushed it)
# !git clone https://github.com/<you>/cs231n-project /content/proj
%cd /content/proj
!pip -q install -r requirements.txt


In [ ]:
# 2. Install HaMeR + checkpoints (~10 min first time; cache to Drive after)
%cd /content
!git clone --recursive https://github.com/geopavlakos/hamer.git
%cd hamer
!pip -q install -e .[all]
!pip -q install -v -e third-party/ViTPose
# download trained models (~2.5GB). If this URL changes, see the HaMeR README.
!bash fetch_demo_data.sh
%cd /content/proj


In [ ]:
# 3. Smoke test: HaMeR loads and runs on one image
import sys; sys.path.insert(0, '/content/proj')
from src.hamer_wrapper import HamerPredictor
import numpy as np, cv2
pred = HamerPredictor()
print('HaMeR loaded OK')
# NOTE: if load_hamer import fails, check hamer's current API —
# adjust the import at the top of src/hamer_wrapper.py accordingly.


In [ ]:
# 4. Point at HO-3D on Drive
HO3D = '/content/drive/MyDrive/HO3D_v3'
from src.ho3d_data import HO3D as HO3DDataset
ds = HO3DDataset(HO3D, split='train')
print(f'{len(ds)} frames available')
fr = ds[0]
print('first frame:', fr.seq, fr.idx, fr.image_path.exists())


In [ ]:
# 5. PILOT: 500 frames end-to-end (expect ~20-40 min on T4)
!python scripts/run_eval.py --ho3d $HO3D --out /content/drive/MyDrive/results_pilot \
    --stage all --max-frames 500 --tta 4


In [ ]:
# 6. Inspect pilot report
import json
print(json.dumps(json.load(open('/content/drive/MyDrive/results_pilot/report.json')), indent=2))


In [ ]:
# 7. FULL RUN (start with 10-20k frames; results cache to Drive, resumable)
!python scripts/run_eval.py --ho3d $HO3D --out /content/drive/MyDrive/results_full \
    --stage all --max-frames 20000 --tta 8


In [ ]:
# 8. Figures
from src import plots
plots.make_all('/content/drive/MyDrive/results_full/scores.npz',
               '/content/drive/MyDrive/results_full/figures')
from IPython.display import Image, display
for f in ['sparsification','occlusion_stratified','filtering','score_vs_error']:
    display(Image(f'/content/drive/MyDrive/results_full/figures/{f}.png'))


## Notes
- **Occlusion masks**: `run_eval.py` reads per-frame masks from `results*/masks/`.
  Render them with `src/ho3d_data.amodal_object_mask` (needs YCB object meshes from the
  HO-3D README) — without them, everything still runs but occlusion stratification is skipped.
- **Resumable**: inference caches per-frame `.npz`; re-running skips finished frames.
- **Colab disconnects**: results live on Drive, so nothing is lost.
